In [4]:
from pathlib import Path
import sys
import inspect

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Находим корень проекта
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# ------------------------------------------------------------
# 2. Импортируем ТВОИ существующие загрузчики EPIC
# ------------------------------------------------------------

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_data import load_split


# ------------------------------------------------------------
# 3. Проверяем, что именно они ожидают
# ------------------------------------------------------------

print("Project root:")
print(PROJECT_ROOT)

print("\nload_baseline_sequences:")
print(inspect.signature(load_baseline_sequences))

print("\nload_split:")
print(inspect.signature(load_split))

Project root:
D:\Projects\EPIC\EPIC

load_baseline_sequences:
(split: 'PreparedSplit') -> 'tuple[dict[str, np.ndarray], dict[str, int]]'

load_split:
(project_root: 'Path | str', config: 'SplitConfig' = SplitConfig(assembly='jaNemVect1.1', split_version='contig_holdout_v1', seed=42, validation_contig_fraction=0.25), *, verify_hashes: 'bool' = True) -> 'PreparedSplit'


In [5]:
import inspect

print("=" * 80)
print("load_split")
print("=" * 80)
print(inspect.getsource(load_split))

print("\n" + "=" * 80)
print("load_baseline_sequences")
print("=" * 80)
print(inspect.getsource(load_baseline_sequences))

load_split
def load_split(
    project_root: Path | str,
    config: SplitConfig = DEFAULT_SPLIT_CONFIG,
    *,
    verify_hashes: bool = True,
) -> PreparedSplit:
    """Загрузить сохранённый split и проверить raw/derived SHA-256."""

    paths = paths_for(project_root, config)
    manifest_path = paths.output / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"Split ещё не подготовлен: {manifest_path}. Вызовите prepare_split()."
        )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_config = {
        "assembly": config.assembly,
        "split_version": config.split_version,
        "seed": config.seed,
        "validation_fraction_of_official_train_contigs": (
            config.validation_contig_fraction
        ),
    }
    for key, expected in expected_config.items():
        if manifest.get(key) != expected:
            raise ValueError(
                f"Manifest {key}={manifest.get(key)!r}, 

In [6]:
import numpy as np

# Загружаем уже подготовленный split
split = load_split(PROJECT_ROOT)

# Загружаем FASTA через твой существующий pipeline
sequences, alphabet = load_baseline_sequences(split)

print("=" * 80)
print("DNA ЗАГРУЖЕНА")
print("=" * 80)

print("Тип sequences:", type(sequences))
print("Количество contig:", len(sequences))
print("Alphabet:", alphabet)

print("\nПервые 5 contig:")
for name, seq in list(sequences.items())[:5]:
    seq = np.asarray(seq)

    print(
        f"{name:25} "
        f"length={len(seq):>12,} "
        f"dtype={seq.dtype} "
        f"unique={np.unique(seq)}"
    )

DNA ЗАГРУЖЕНА
Тип sequences: <class 'dict'>
Количество contig: 47
Alphabet: {'A': 54173800, 'C': 39020211, 'G': 38944862, 'N': 34600, 'T': 54153761, 'a': 25666946, 'c': 15841270, 'g': 15877204, 'n': 600, 't': 25705184}

Первые 5 contig:
NC_064034.1               length=  22,168,631 dtype=uint8 unique=[0 1 2 3 4 5 6 7 8]
NC_064035.1               length=  22,031,953 dtype=uint8 unique=[0 1 2 3 4 5 6 7 8]
NC_064036.1               length=  20,793,841 dtype=uint8 unique=[0 1 2 3 4 5 6 7 8]
NC_064037.1               length=  19,319,694 dtype=uint8 unique=[0 1 2 3 4 5 6 7 8]
NC_064038.1               length=  19,000,648 dtype=uint8 unique=[0 1 2 3 4 5 6 7 8]


In [7]:
import inspect

read_fasta = load_baseline_sequences.__globals__["read_fasta"]

print("=" * 80)
print("read_fasta")
print("=" * 80)
print(inspect.getsource(read_fasta))

read_fasta
def read_fasta(path: Path | str) -> tuple[dict[str, np.ndarray], dict[str, int]]:
    """Read a plain or gzip FASTA into compact, case-preserving arrays."""

    fasta_path = Path(path)
    opener = gzip.open if fasta_path.suffix.lower() == ".gz" else open
    sequences: dict[str, np.ndarray] = {}
    alphabet = np.zeros(256, dtype=np.int64)
    name: str | None = None
    chunks: list[bytes] = []

    def flush() -> None:
        nonlocal name, chunks
        if name is None:
            return
        if name in sequences:
            raise ValueError(f"Duplicate FASTA identifier: {name}")
        payload = b"".join(chunks)
        if not payload:
            raise ValueError(f"Empty FASTA record: {name}")
        raw = np.frombuffer(payload, dtype=np.uint8)
        alphabet[:] += np.bincount(raw, minlength=256)
        sequences[name] = FASTA_LOOKUP[raw]

    with opener(fasta_path, "rb") as stream:
        for line in stream:
            if line.startswith(b">"):
       

In [8]:
FASTA_LOOKUP = read_fasta.__globals__["FASTA_LOOKUP"]

symbols = "ACGTacgtnN"

print("=" * 60)
print("FASTA ENCODING")
print("=" * 60)

for ch in symbols:
    code = int(FASTA_LOOKUP[ord(ch)])
    print(f"{ch!r:>4} -> {code}")

FASTA ENCODING
 'A' -> 0
 'C' -> 1
 'G' -> 2
 'T' -> 3
 'a' -> 4
 'c' -> 5
 'g' -> 6
 't' -> 7
 'n' -> 8
 'N' -> 8


In [9]:
assignment = split.contig_assignment

print("=" * 80)
print("CONTIG ASSIGNMENT")
print("=" * 80)

print("Columns:")
print(list(assignment.columns))

print("\nShape:")
print(assignment.shape)

print("\nПервые строки:")
display(assignment.head(10))

print("\nУникальные значения по колонкам:")
for column in assignment.columns:
    if assignment[column].nunique() < 20:
        print(f"\n{column}:")
        print(assignment[column].value_counts(dropna=False))

CONTIG ASSIGNMENT
Columns:
['contig', 'split', 'source_split']

Shape:
(15, 3)

Первые строки:


,contig,split,source_split
0,NC_064034.1,validation,train
1,NC_064035.1,train,train
2,NC_064036.1,train,train
3,NC_064038.1,train,train
4,NC_064039.1,train,train
5,NC_064040.1,train,train
6,NC_064041.1,validation,train
7,NC_064042.1,validation,train
8,NC_064043.1,train,train
9,NC_064045.1,train,train



Уникальные значения по колонкам:

contig:
contig
NC_064034.1    1
NC_064035.1    1
NC_064036.1    1
NC_064038.1    1
NC_064039.1    1
NC_064040.1    1
NC_064041.1    1
NC_064042.1    1
NC_064043.1    1
NC_064045.1    1
NC_064047.1    1
NC_064048.1    1
NC_064037.1    1
NC_064044.1    1
NC_064046.1    1
Name: count, dtype: int64

split:
split
train         9
validation    3
test          3
Name: count, dtype: int64

source_split:
source_split
train    12
test      3
Name: count, dtype: int64


In [10]:
import numpy as np

# Точная кодировка, которую мы только что проверили
ID_TO_SYMBOL = {
    0: "A",
    1: "C",
    2: "G",
    3: "T",
    4: "a",
    5: "c",
    6: "g",
    7: "t",
    8: "N",
}

VOCAB_SIZE = 9


# ------------------------------------------------------------
# Берём ТОЛЬКО train-contigs
# ------------------------------------------------------------

train_contigs = (
    split.contig_assignment
    .loc[split.contig_assignment["split"] == "train", "contig"]
    .astype(str)
    .tolist()
)

print("=" * 80)
print("TRAIN CONTIGS")
print("=" * 80)

total_bp = 0

for contig in train_contigs:
    seq = sequences[contig]
    total_bp += len(seq)
    print(f"{contig:20} {len(seq):>15,} bp")

print("-" * 80)
print(f"Количество train-contigs: {len(train_contigs)}")
print(f"Всего train bp:           {total_bp:,}")


# ------------------------------------------------------------
# Считаем все 9 × 9 = 81 возможную пару
# ------------------------------------------------------------

pair_counts = np.zeros(VOCAB_SIZE * VOCAB_SIZE, dtype=np.int64)

for contig in train_contigs:
    seq = sequences[contig]

    # Каждую пару (left, right) кодируем одним числом:
    #
    # pair_id = left * 9 + right
    #
    # Например:
    # A,A = 0*9 + 0 = 0
    # A,C = 0*9 + 1 = 1
    # T,G = 3*9 + 2 = 29

    pair_ids = seq[:-1] * VOCAB_SIZE + seq[1:]

    pair_counts += np.bincount(
        pair_ids,
        minlength=VOCAB_SIZE * VOCAB_SIZE
    )


# ------------------------------------------------------------
# TOP-10
# ------------------------------------------------------------

top_indices = np.argsort(pair_counts)[::-1][:10]

total_pairs = pair_counts.sum()

print("\n" + "=" * 80)
print("TOP-10 СОСЕДНИХ ПАР В TRAIN")
print("=" * 80)

for rank, pair_id in enumerate(top_indices, start=1):

    left = pair_id // VOCAB_SIZE
    right = pair_id % VOCAB_SIZE

    count = pair_counts[pair_id]
    share = count / total_pairs * 100

    pair_text = ID_TO_SYMBOL[left] + ID_TO_SYMBOL[right]

    print(
        f"{rank:>2}. "
        f"{pair_text:<4} "
        f"({left},{right}) "
        f"count={count:>12,} "
        f"share={share:>7.3f}%"
    )

print("\nВсего соседних пар:", f"{total_pairs:,}")

TRAIN CONTIGS
NC_064035.1               22,031,953 bp
NC_064036.1               20,793,841 bp
NC_064038.1               19,000,648 bp
NC_064039.1               18,096,078 bp
NC_064040.1               17,865,913 bp
NC_064043.1               16,593,782 bp
NC_064045.1               16,388,311 bp
NC_064047.1               15,372,318 bp
NC_064048.1               13,717,364 bp
--------------------------------------------------------------------------------
Количество train-contigs: 9
Всего train bp:           159,860,208

TOP-10 СОСЕДНИХ ПАР В TRAIN
 1. AA   (0,0) count=  10,046,053 share=  6.284%
 2. TT   (3,3) count=  10,034,383 share=  6.277%
 3. AT   (0,3) count=   8,631,341 share=  5.399%
 4. TA   (3,0) count=   7,794,418 share=  4.876%
 5. CA   (1,0) count=   7,514,233 share=  4.701%
 6. TG   (3,2) count=   7,487,859 share=  4.684%
 7. CT   (1,3) count=   6,754,577 share=  4.225%
 8. AG   (0,2) count=   6,728,039 share=  4.209%
 9. TC   (3,1) count=   6,536,022 share=  4.089%
10. GA   

In [11]:
import numpy as np

# ------------------------------------------------------------
# Рабочая копия ТОЛЬКО train-contigs для BPE
# ------------------------------------------------------------

bpe_sequences = {
    contig: sequences[contig].astype(np.uint16, copy=True)
    for contig in train_contigs
}

# Исходный словарь
token_text = {
    0: "A",
    1: "C",
    2: "G",
    3: "T",
    4: "a",
    5: "c",
    6: "g",
    7: "t",
    8: "N",
}

original_bp = sum(len(seq) for seq in bpe_sequences.values())


# ------------------------------------------------------------
# Non-overlapping merge одной пары
# ------------------------------------------------------------

def merge_pair(seq, left, right, new_token):
    """
    Left-to-right non-overlapping BPE merge.

    Например:
        A A A A
    превращается в:
        X X
    а не пытается использовать перекрывающиеся AA.
    """

    match = (seq[:-1] == left) & (seq[1:] == right)

    if not match.any():
        return seq, 0

    starts = np.flatnonzero(match)

    # Для одинаковых токенов возможны перекрытия:
    # AAAA -> match на позициях 0,1,2.
    #
    # Оставляем 0,2.
    if left == right:
        # Разбиваем последовательные match-позиции на группы
        new_group = np.r_[True, np.diff(starts) != 1]
        group_id = np.cumsum(new_group) - 1

        # Позиция внутри каждой группы
        group_starts = np.flatnonzero(new_group)
        offsets = np.arange(len(starts)) - np.repeat(
            group_starts,
            np.diff(np.r_[group_starts, len(starts)])
        )

        starts = starts[offsets % 2 == 0]

    merged_count = len(starts)

    # Первый элемент пары заменяем новым токеном
    out = seq.copy()
    out[starts] = new_token

    # Второй элемент каждой merged-пары удаляем
    remove = np.zeros(len(seq), dtype=bool)
    remove[starts + 1] = True

    out = out[~remove]

    return out, merged_count


# ------------------------------------------------------------
# Первый BPE merge
# ------------------------------------------------------------

LEFT = 0       # A
RIGHT = 0      # A
NEW_TOKEN = 9

token_text[NEW_TOKEN] = (
    token_text[LEFT] + token_text[RIGHT]
)

print("=" * 80)
print("BPE ITERATION 1")
print("=" * 80)

print(
    f"Выбранная пара: "
    f"{token_text[LEFT]} + {token_text[RIGHT]}"
)
print(
    f"Новый токен: #{NEW_TOKEN} = "
    f"{token_text[NEW_TOKEN]}"
)

total_merged = 0

for contig in train_contigs:

    before = len(bpe_sequences[contig])

    merged_seq, merged_count = merge_pair(
        bpe_sequences[contig],
        LEFT,
        RIGHT,
        NEW_TOKEN,
    )

    bpe_sequences[contig] = merged_seq

    total_merged += merged_count

    print(
        f"{contig:20} "
        f"{before:>12,} -> {len(merged_seq):>12,} "
        f"| merges={merged_count:>10,}"
    )


# ------------------------------------------------------------
# Итог
# ------------------------------------------------------------

new_total_tokens = sum(
    len(seq)
    for seq in bpe_sequences.values()
)

compression = original_bp / new_total_tokens

print("\n" + "-" * 80)

print(f"Исходных bp:       {original_bp:,}")
print(f"Сделано merge:     {total_merged:,}")
print(f"Осталось токенов:  {new_total_tokens:,}")
print(f"Compression ratio: {compression:.4f}x")
print(f"Среднее bp/token:  {compression:.4f}")

BPE ITERATION 1
Выбранная пара: A + A
Новый токен: #9 = AA
NC_064035.1            22,031,953 ->   20,897,736 | merges= 1,134,217
NC_064036.1            20,793,841 ->   19,744,261 | merges= 1,049,580
NC_064038.1            19,000,648 ->   18,214,504 | merges=   786,144
NC_064039.1            18,096,078 ->   17,177,205 | merges=   918,873
NC_064040.1            17,865,913 ->   17,018,192 | merges=   847,721
NC_064043.1            16,593,782 ->   15,801,999 | merges=   791,783
NC_064045.1            16,388,311 ->   15,713,835 | merges=   674,476
NC_064047.1            15,372,318 ->   14,659,847 | merges=   712,471
NC_064048.1            13,717,364 ->   13,117,608 | merges=   599,756

--------------------------------------------------------------------------------
Исходных bp:       159,860,208
Сделано merge:     7,515,021
Осталось токенов:  152,345,187
Compression ratio: 1.0493x
Среднее bp/token:  1.0493


In [12]:
import numpy as np


# Рабочая копия TRAIN
bpe_sequences = {
    contig: sequences[contig].astype(np.uint16, copy=True)
    for contig in train_contigs
}

token_text = {
    0: "A",
    1: "C",
    2: "G",
    3: "T",
    4: "a",
    5: "c",
    6: "g",
    7: "t",
    8: "N",
}

original_bp = sum(len(x) for x in bpe_sequences.values())


def merge_pair(seq, left, right, new_token):
    match = (seq[:-1] == left) & (seq[1:] == right)

    starts = np.flatnonzero(match)

    if len(starts) == 0:
        return seq, 0

    # Перекрытия возможны только для X+X:
    # AAAA имеет AA в 0,1,2, но merge должны быть 0 и 2.
    if left == right:
        keep = np.ones(len(starts), dtype=bool)

        last = -2
        for i, pos in enumerate(starts):
            if pos == last + 1:
                keep[i] = False
            else:
                last = pos

        starts = starts[keep]

    out = seq.copy()
    out[starts] = new_token

    remove = np.zeros(len(seq), dtype=bool)
    remove[starts + 1] = True

    return out[~remove], len(starts)


def count_all_pairs(sequences, vocab_size):
    counts = np.zeros(vocab_size * vocab_size, dtype=np.int64)

    for seq in sequences.values():
        pair_ids = (
            seq[:-1].astype(np.int64) * vocab_size
            + seq[1:]
        )

        counts += np.bincount(
            pair_ids,
            minlength=vocab_size * vocab_size
        )

    return counts


NUM_MERGES = 10   # пока только 10 для первого теста

for iteration in range(1, NUM_MERGES + 1):

    vocab_size = max(token_text) + 1

    counts = count_all_pairs(
        bpe_sequences,
        vocab_size
    )

    top = np.argsort(counts)[::-1][:10]

    print("\n" + "=" * 90)
    print(f"BPE ITERATION {iteration}")
    print("=" * 90)

    for rank, pair_id in enumerate(top, 1):

        left = pair_id // vocab_size
        right = pair_id % vocab_size

        print(
            f"{rank:>2}. "
            f"{token_text[left]} + {token_text[right]} "
            f"→ {token_text[left] + token_text[right]:<20} "
            f"| count={counts[pair_id]:,}"
        )

    # Самая частая пара
    best = top[0]

    left = best // vocab_size
    right = best % vocab_size

    new_token = vocab_size
    token_text[new_token] = (
        token_text[left] + token_text[right]
    )

    total_merges = 0

    for contig in train_contigs:

        seq, n = merge_pair(
            bpe_sequences[contig],
            left,
            right,
            new_token
        )

        bpe_sequences[contig] = seq
        total_merges += n

    total_tokens = sum(
        len(x) for x in bpe_sequences.values()
    )

    compression = original_bp / total_tokens

    print("\nSELECTED:")
    print(
        f"{token_text[left]} + {token_text[right]}"
        f" → TOKEN {new_token} = {token_text[new_token]}"
    )

    print(f"Actual merges:      {total_merges:,}")
    print(f"Tokens remaining:   {total_tokens:,}")
    print(f"Compression:        {compression:.4f}x")
    print(f"Average bp/token:   {compression:.4f}")


BPE ITERATION 1
 1. A + A → AA                   | count=10,046,053
 2. T + T → TT                   | count=10,034,383
 3. A + T → AT                   | count=8,631,341
 4. T + A → TA                   | count=7,794,418
 5. C + A → CA                   | count=7,514,233
 6. T + G → TG                   | count=7,487,859
 7. C + T → CT                   | count=6,754,577
 8. A + G → AG                   | count=6,728,039
 9. T + C → TC                   | count=6,536,022
10. G + A → GA                   | count=6,514,480

SELECTED:
A + A → TOKEN 9 = AA
Actual merges:      7,515,021
Tokens remaining:   152,345,187
Compression:        1.0493x
Average bp/token:   1.0493

BPE ITERATION 2
 1. T + T → TT                   | count=10,034,383
 2. T + G → TG                   | count=7,487,859
 3. C + T → CT                   | count=6,754,577
 4. A + T → AT                   | count=6,741,353
 5. T + C → TC                   | count=6,536,022
 6. G + T → GT                   | count=6,435,88

In [13]:
vocab_size = max(token_text) + 1

counts = count_all_pairs(
    bpe_sequences,
    vocab_size
)

pairs_with_bpe = []

for pair_id, count in enumerate(counts):

    if count == 0:
        continue

    left = pair_id // vocab_size
    right = pair_id % vocab_size

    # Хотя бы один токен уже создан BPE
    if left >= 9 or right >= 9:
        pairs_with_bpe.append(
            (count, left, right)
        )

pairs_with_bpe.sort(reverse=True)

print("=" * 90)
print("TOP-20 ПАР, ГДЕ ЕСТЬ BPE TOKEN")
print("=" * 90)

for rank, (count, left, right) in enumerate(
    pairs_with_bpe[:20],
    1
):
    print(
        f"{rank:>2}. "
        f"{token_text[left]:<15} + "
        f"{token_text[right]:<15} "
        f"→ {token_text[left] + token_text[right]:<30} "
        f"| count={count:>12,}"
    )

TOP-20 ПАР, ГДЕ ЕСТЬ BPE TOKEN
 1. T               + AA              → TAA                            | count=   1,828,416
 2. AA              + A               → AAA                            | count=   1,733,188
 3. C               + AA              → CAA                            | count=   1,515,930
 4. C               + TT              → CTT                            | count=   1,376,274
 5. TT              + G               → TTG                            | count=   1,371,657
 6. G               + AA              → GAA                            | count=   1,298,218
 7. AA              + G               → AAG                            | count=   1,264,765
 8. TT              + A               → TTA                            | count=   1,238,096
 9. CA              + G               → CAG                            | count=   1,224,528
10. G               + TT              → GTT                            | count=   1,135,931
11. TG              + A               → TGA      

In [14]:
import numpy as np
from collections import Counter

# ------------------------------------------------------------
# Фиксируем уже выполненные 10 merge в правильном порядке
# ------------------------------------------------------------

merge_history = [
    (0, 0, 9),    # A  + A  -> AA
    (3, 3, 10),   # T  + T  -> TT
    (3, 2, 11),   # T  + G  -> TG
    (7, 7, 12),   # t  + t  -> tt
    (4, 4, 13),   # a  + a  -> aa
    (1, 0, 14),   # C  + A  -> CA
    (3, 0, 15),   # T  + A  -> TA
    (3, 1, 16),   # T  + C  -> TC
    (2, 2, 17),   # G  + G  -> GG
    (1, 1, 18),   # C  + C  -> CC
]

TARGET_MERGES = 50
TOP_K = 10

# ------------------------------------------------------------
# Продолжаем с текущего состояния bpe_sequences
# ------------------------------------------------------------

current_iteration = len(merge_history) + 1

while len(merge_history) < TARGET_MERGES:

    vocab_size = max(token_text) + 1

    counts = count_all_pairs(
        bpe_sequences,
        vocab_size
    )

    top_ids = np.argsort(counts)[::-1][:TOP_K]

    print("\n" + "=" * 100)
    print(f"BPE ITERATION {current_iteration}")
    print("=" * 100)

    for rank, pair_id in enumerate(top_ids, start=1):

        left = int(pair_id // vocab_size)
        right = int(pair_id % vocab_size)

        left_text = token_text[left]
        right_text = token_text[right]

        combined = left_text + right_text

        print(
            f"{rank:>2}. "
            f"{left_text:<15} + "
            f"{right_text:<15} "
            f"→ {combined:<30} "
            f"| count={counts[pair_id]:>12,} "
            f"| len={len(combined):>3} bp"
        )

    # --------------------------------------------------------
    # Берём самую частую пару
    # --------------------------------------------------------

    best_pair_id = int(top_ids[0])

    left = best_pair_id // vocab_size
    right = best_pair_id % vocab_size

    new_token = vocab_size

    token_text[new_token] = (
        token_text[left]
        + token_text[right]
    )

    # --------------------------------------------------------
    # Применяем merge ко всем train-contigs
    # --------------------------------------------------------

    total_merges = 0

    for contig in train_contigs:

        merged_seq, merged_count = merge_pair(
            bpe_sequences[contig],
            left,
            right,
            new_token,
        )

        bpe_sequences[contig] = merged_seq
        total_merges += merged_count

    merge_history.append(
        (left, right, new_token)
    )

    # --------------------------------------------------------
    # Статистика
    # --------------------------------------------------------

    total_tokens = sum(
        len(seq)
        for seq in bpe_sequences.values()
    )

    compression = original_bp / total_tokens

    print("\nSELECTED:")
    print(
        f"{token_text[left]} + {token_text[right]}"
        f" → TOKEN #{new_token}"
        f" = {token_text[new_token]}"
    )

    print(f"Token length:        {len(token_text[new_token])} bp")
    print(f"Actual merges:       {total_merges:,}")
    print(f"Tokens remaining:    {total_tokens:,}")
    print(f"Compression ratio:   {compression:.4f}x")
    print(f"Average bp/token:    {compression:.4f}")

    current_iteration += 1


# ============================================================
# ФИНАЛЬНАЯ СТАТИСТИКА ПОСЛЕ 50 MERGES
# ============================================================

total_tokens = sum(
    len(seq)
    for seq in bpe_sequences.values()
)

compression = original_bp / total_tokens

length_distribution = Counter(
    len(text)
    for token_id, text in token_text.items()
    if token_id >= 9
)

print("\n" + "#" * 100)
print("BPE SUMMARY")
print("#" * 100)

print(f"Всего merge rules:       {len(merge_history)}")
print(f"Размер словаря:          {len(token_text)}")
print(f"Исходных bp:             {original_bp:,}")
print(f"Итоговых BPE tokens:     {total_tokens:,}")
print(f"Compression ratio:       {compression:.4f}x")
print(f"Среднее bp/token:        {compression:.4f}")

print("\nРАСПРЕДЕЛЕНИЕ ДЛИНЫ СОЗДАННЫХ ТОКЕНОВ:")

for length in sorted(length_distribution):
    print(
        f"{length:>3} bp : "
        f"{length_distribution[length]:>4} tokens"
    )

print("\nСАМЫЕ ДЛИННЫЕ ТОКЕНЫ:")

longest = sorted(
    [
        (len(text), token_id, text)
        for token_id, text in token_text.items()
        if token_id >= 9
    ],
    reverse=True
)[:20]

for length, token_id, text in longest:
    print(
        f"#{token_id:<4} "
        f"{text:<35} "
        f"{length:>3} bp"
    )


BPE ITERATION 11
 1. G               + A               → GA                             | count=   2,811,437 | len=  2 bp
 2. c               + c               → cc                             | count=   2,395,490 | len=  2 bp
 3. t               + g               → tg                             | count=   2,375,872 | len=  2 bp
 4. g               + g               → gg                             | count=   2,358,466 | len=  2 bp
 5. a               + t               → at                             | count=   2,170,807 | len=  2 bp
 6. a               + c               → ac                             | count=   2,164,240 | len=  2 bp
 7. t               + a               → ta                             | count=   2,119,812 | len=  2 bp
 8. c               + a               → ca                             | count=   2,119,382 | len=  2 bp
 9. G               + C               → GC                             | count=   1,978,805 | len=  2 bp
10. a               + g              